In [46]:
import numpy as np
import pandas as pd
import gc

In [47]:
df_train = pd.read_csv("dataset/train.csv")
df_test = pd.read_csv("dataset/test.csv")

In [48]:
categorical_columns = df_train.select_dtypes(include=["object"]).columns
unique_values = {col: df_train[col].nunique() for col in categorical_columns}

for col, unique_count in unique_values.items():
  print(f"{col}: {unique_count} unique values")

print("\n")

categorical_columns = df_test.select_dtypes(include=["object"]).columns
unique_values = {col: df_test[col].nunique() for col in categorical_columns}

for col, unique_count in unique_values.items():
  print(f"{col}: {unique_count} unique values")
  
gc.collect()

Soil Type: 5 unique values
Crop Type: 11 unique values
Fertilizer Name: 7 unique values


Soil Type: 5 unique values
Crop Type: 11 unique values


63

In [49]:
import seaborn as sns

In [50]:
missing_threshold = 0.95

high_missing_columns = df_train.columns[df_train.isnull().mean() > missing_threshold]

df_train = df_train.drop(columns=high_missing_columns)
df_test = df_test.drop(columns=high_missing_columns)

for column in df_train.columns:
  if df_train[column].isnull().any():
    if df_train[column].dtype == "object":
      mode_value = df_train[column].mode()[0]
      df_train[column].fillna(mode_value, inplace=True)
      df_test[column].fillna(mode_value, inplace=True)
    else:
      median_value = df_train[column].median()
      df_train[column].fillna(median_value, inplace=True)
      df_test[column].fillna(median_value, inplace=True)

In [53]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

cat_cols_train = df_train.select_dtypes(include="object").columns
cat_cols_test = cat_cols_train[cat_cols_train != "Fertilizer Name"]

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

df_train[cat_cols_train] = ordinal_encoder.fit_transform(df_train[cat_cols_train].astype(str))
df_test[cat_cols_test] = ordinal_encoder.transform(df_test[cat_cols_test].astype(str))

In [70]:
le = LabelEncoder()
df_train["Fertilizer Name"] = le.fit_transform(df_train["Fertilizer Name"])

In [55]:
y = df_train["Fertilizer Name"]
X = df_train.drop(["Fertilizer Name"], axis=1)

In [56]:
from sklearn.model_selection import train_test_split
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [57]:
from xgboost import XGBClassifier
model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(train_y)),
    n_estimators=3200,
    learning_rate=0.045,         
    max_depth=7,                
    colsample_bytree=0.6,       
    colsample_bylevel=0.8,      
    subsample=0.8,
)

model.fit(train_X, train_y)
y_pred_probs = model.predict_proba(test_X)
top_3_preds = np.argsort(y_pred_probs, axis=1)[:, -3:][:, ::-1]
actual = [[label] for label in test_y]

def mapk(actual, predicted, k=3):
  def apk(a, p, k):
    p = p[:k]
    score = 0.0
    hits = 0
    seen = set()
    for i, pred in enumerate(p):
      if pred in a and pred not in seen:
        hits += 1
        score += hits / (i+1.0)
        seen.add(pred)
    return score / min(len(a), k)
  return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

map3_score = mapk(actual, top_3_preds)
print(f"score: {map3_score:.5f}")

score: 0.33531


In [65]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

df_train = pd.read_csv("dataset/train.csv")
df_test = pd.read_csv("dataset/test.csv")

cat_cols_train = df_train.select_dtypes(include="object").columns
cat_cols_test = cat_cols_train[cat_cols_train != "Fertilizer Name"]

print(cat_cols_test)

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

df_train[cat_cols_test] = ordinal_encoder.fit_transform(df_train[cat_cols_test].astype(str))
df_test[cat_cols_test] = ordinal_encoder.transform(df_test[cat_cols_test].astype(str))

Index(['Soil Type', 'Crop Type'], dtype='object')


In [72]:
test_probs = model.predict_proba(df_test)
top_3_preds = np.argsort(test_probs, axis=1)[:, -3:][:, ::-1]
top_3_labels = le.inverse_transform(top_3_preds.ravel()).reshape(top_3_preds.shape)

df_sub = pd.read_csv("dataset/sample_submission.csv")

submission = pd.DataFrame({
    'id': df_sub['id'],
    'Fertilizer Name': [' '.join(row) for row in top_3_labels]
})
submission.to_csv('submission.csv', index=False)

[['20-20' 'DAP' '28-28']
 ['17-17-17' '14-35-14' 'Urea']
 ['20-20' 'Urea' '10-26-26']
 ...
 ['14-35-14' 'DAP' '17-17-17']
 ['17-17-17' '10-26-26' 'DAP']
 ['14-35-14' '17-17-17' '20-20']]
